# 03 — Baseline Models and LSTM Training

**Project:** Agentic Campaign Strategist — Audience-to-Campaign Fit Advisor  
**Team:** Matt Hashemi, Marwah Faraj, Rebecca Cloe (AAI-590, Group 5)  
**Author of this notebook:** TODO  

**Purpose.** Train traditional ML baselines (KNN, Random Forest) and a PyTorch LSTM from scratch on the weekly journey sequences, with a leakage-safe evaluation design:

1. **Grouped split by household** — no household appears in both train and test.
2. **Train-only scaling** — the scaler is fit on training data only.
3. **Class weights** — to handle label imbalance.
4. **Train/validation curves** — to show fit quality, as required by the report template.

## 1. Setup

In [ ]:
# Setup: make the src/ package importable.
# Run once from the repo root if not installed yet:  pip install -e .
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.append(str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 7
np.random.seed(RANDOM_STATE)

## 2. Load Model-Ready Data

In [ ]:
from campaign_strategist.config import PROCESSED_DATA_DIR, CAMPAIGN_CLASSES

x = np.load(PROCESSED_DATA_DIR / 'sequences_x.npy')
y = np.load(PROCESSED_DATA_DIR / 'labels_y.npy')
sample_index = pd.read_parquet(PROCESSED_DATA_DIR / 'sample_index.parquet')
print(x.shape, y.shape)

## 3. Grouped Train/Validation/Test Split

We split **by household** so overlapping windows from one household cannot leak across splits.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

groups = sample_index['household_id'].to_numpy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_val_idx, test_idx = next(gss.split(x, y, groups=groups))

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_STATE)
train_idx_rel, val_idx_rel = next(gss_val.split(x[train_val_idx], y[train_val_idx], groups=groups[train_val_idx]))
train_idx, val_idx = train_val_idx[train_idx_rel], train_val_idx[val_idx_rel]

assert not set(groups[train_idx]) & set(groups[test_idx]), 'Household leakage between train and test!'
print(f'train {len(train_idx):,} | val {len(val_idx):,} | test {len(test_idx):,}')

## 4. Train-Only Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
n_feat = x.shape[-1]
scaler.fit(x[train_idx].reshape(-1, n_feat))

def scale(a):
    return scaler.transform(a.reshape(-1, n_feat)).reshape(a.shape).astype(np.float32)

x_train, x_val, x_test = scale(x[train_idx]), scale(x[val_idx]), scale(x[test_idx])
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

## 5. Baseline Models (KNN and Random Forest)

Sequences are flattened into summary features (mean, last step, trend) for the tabular baselines. *Discussion (TODO): what information do baselines lose vs. the sequence model?*

In [ ]:
from campaign_strategist.baselines import flatten_sequences
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

f_train, f_val, f_test = (flatten_sequences(a) for a in (x_train, x_val, x_test))

baselines = {
    'knn': KNeighborsClassifier(n_neighbors=15, weights='distance'),
    'random_forest': RandomForestClassifier(n_estimators=300, max_depth=12,
                                            class_weight='balanced', random_state=RANDOM_STATE),
}
baseline_results = {}
for name, model in baselines.items():
    model.fit(f_train, y_train)
    pred = model.predict(f_test)
    baseline_results[name] = {'accuracy': accuracy_score(y_test, pred),
                              'macro_f1': f1_score(y_test, pred, average='macro')}
    print(name, baseline_results[name])

## 6. LSTM Model — Trained from Scratch

*TODO: describe the architecture here (hidden size, layers, classifier head, dropout) in words for the report methodology section.*

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from campaign_strategist.model import JourneyLSTM

torch.manual_seed(RANDOM_STATE)

class_counts = np.bincount(y_train, minlength=len(CAMPAIGN_CLASSES))
class_weights = torch.tensor(len(y_train) / (len(CAMPAIGN_CLASSES) * np.maximum(class_counts, 1)),
                             dtype=torch.float32)

model = JourneyLSTM(n_features=x.shape[-1], hidden_size=48)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

train_loader = DataLoader(TensorDataset(torch.tensor(x_train), torch.tensor(y_train)),
                          batch_size=64, shuffle=True)

EPOCHS = 30
history = []
for epoch in range(EPOCHS):
    model.train(); losses = []
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(bx), by)
        loss.backward(); optimizer.step()
        losses.append(loss.item())
    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(torch.tensor(x_val)), torch.tensor(y_val)).item()
    history.append({'epoch': epoch + 1, 'train_loss': float(np.mean(losses)), 'val_loss': val_loss})
    print(history[-1])

## 7. Train vs. Validation Curves

*Discussion (TODO): does the model overfit or underfit? Where does validation loss flatten?*

In [ ]:
hist = pd.DataFrame(history)
plt.plot(hist.epoch, hist.train_loss, label='train loss')
plt.plot(hist.epoch, hist.val_loss, label='validation loss')
plt.xlabel('Epoch'); plt.ylabel('Cross-entropy loss'); plt.legend()
plt.title('LSTM training and validation loss')
plt.tight_layout(); plt.show()

## 8. Test-Set Evaluation and Comparison

Macro-F1 is the headline metric because of class imbalance.

In [ ]:
model.eval()
with torch.no_grad():
    lstm_pred = model(torch.tensor(x_test)).argmax(dim=1).numpy()

results = pd.DataFrame([
    {'model': 'LSTM', 'accuracy': accuracy_score(y_test, lstm_pred),
     'macro_f1': f1_score(y_test, lstm_pred, average='macro')},
    *[{'model': k, **v} for k, v in baseline_results.items()],
]).sort_values('macro_f1', ascending=False)
results

## 9. Save Artifacts for Notebook 04

In [ ]:
# from campaign_strategist.model import save_model
# save_model(model, str(REPO_ROOT / 'artifacts' / 'journey_lstm.pt'))
# np.save(..., test indices etc.)

## 10. Summary

*TODO: 3-5 sentences on findings.*

## References

*TODO: cite PyTorch, scikit-learn, LSTM (Hochreiter & Schmidhuber, 1997).*